In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("FactSales.csv")
df.head()

In [ ]:
df = df.rename(columns={
    'Value ( Quantity * Rate )':'Revenue'
})
df.columns

In [ ]:
df['Date'] = pd.to_datetime(
    df['Date'],
    format='%d-%b-%y'
)

In [ ]:
df.info()

In [ ]:
#setup and validation
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
pd.set_option('display.max_columns', None)

df['Date'] = pd.to_datetime(df['Date'])

print("Shape of dataset:", df.shape)
print("\nDate range:", df['Date'].min().date(), "to", df['Date'].max().date())
print("\nUnique customers:", df['Customer'].nunique())
print("\nUnique products:", df['Products'].nunique())

df.head()

In [ ]:
#setup and validation
key_cols = ['Date', 'Customer', 'Products', 'Quantity', 'Rate', 'Revenue']

print("Missing values per column:\n", df[key_cols].isnull().sum())

revenue_check = (df['Quantity'] * df['Rate']).round(2) == df['Revenue'].round(2)

print(
    "\nRows where Revenue = Quantity x Rate holds true:",
    revenue_check.sum(),
    "out of",
    len(df)
)

In [ ]:
#RFM Analysis
snapshot_date = df['Date'].max() + pd.Timedelta(days=1)

print(
    "Snapshot date used for Recency calculation:",
    snapshot_date.date()
)

rfm = df.groupby('Customer').agg(
    Recency=('Date',
             lambda x:
             (snapshot_date - x.max()).days),

    Frequency=('Date', 'nunique'),

    Monetary=('Revenue', 'sum')
).reset_index()

rfm.sort_values(
    'Monetary',
    ascending=False
).head(10)

In [ ]:
#RFM Analysis
rfm['R_Score'] = pd.qcut(
    rfm['Recency'],
    4,
    labels=[4,3,2,1]
).astype(int)

rfm['F_Score'] = pd.qcut(
    rfm['Frequency'].rank(method='first'),
    4,
    labels=[1,2,3,4]
).astype(int)

rfm['M_Score'] = pd.qcut(
    rfm['Monetary'],
    4,
    labels=[1,2,3,4]
).astype(int)

rfm['RFM_Score'] = (
    rfm['R_Score'].astype(str)
    +
    rfm['F_Score'].astype(str)
    +
    rfm['M_Score'].astype(str)
)

rfm.head(10)

In [ ]:
#RFM Analysis
def assign_segment(row):

    if row['R_Score'] >= 4 and row['F_Score'] >= 4 and row['M_Score'] >= 4:
        return 'Champions'

    elif row['F_Score'] >= 3 and row['M_Score'] >= 3:
        return 'Loyal Customers'

    elif row['R_Score'] <= 2 and row['F_Score'] >= 3:
        return 'At Risk'

    elif row['R_Score'] <= 2 and row['F_Score'] <= 2:
        return 'Lost Customers'

    else:
        return 'Potential / Others'

rfm['Segment'] = rfm.apply(
    assign_segment,
    axis=1
)

In [ ]:
#RFM Analysis
segment_summary = rfm.groupby(
    'Segment'
).agg(
    Customers=('Customer', 'count'),
    Total_Revenue=('Monetary', 'sum')
)

segment_summary

In [ ]:
#RFM Analysis
fig, axes = plt.subplots(1,2,figsize=(12,5))

segment_summary['Customers'].plot(
    kind='bar',
    ax=axes[0]
)

segment_summary['Total_Revenue'].plot(
    kind='bar',
    ax=axes[1]
)

plt.tight_layout()
plt.savefig("RFM_Segments.png", bbox_inches="tight")
plt.show()

In [ ]:
#Pareto Analysis
cust_rev = (
    df.groupby('Customer')['Revenue']
      .sum()
      .sort_values(ascending=False)
      .reset_index()
)

cust_rev['Revenue_%'] = (
    cust_rev['Revenue']
    /
    cust_rev['Revenue'].sum()
) * 100

cust_rev['Cumulative_%'] = (
    cust_rev['Revenue_%']
    .cumsum()
)

cust_rev.head(10)

In [ ]:
#Pareto Analysis
customers_to_80 = cust_rev[
    cust_rev['Cumulative_%'] <= 80
]

num_customers_80 = len(customers_to_80) + 1

pct_of_customer_base = (
    num_customers_80
    /
    len(cust_rev)
) * 100

print(
    f"Customers needed to reach 80% of total revenue: {num_customers_80}"
)

print(
    f"That is {pct_of_customer_base:.1f}% of the customer base."
)

In [ ]:
#Pareto Analysis
fig, ax1 = plt.subplots(figsize=(12,6))

top_n = 10
plot_data = cust_rev.head(top_n).copy()

plot_data['Rank'] = range(1, top_n + 1)

# Revenue bars
ax1.bar(
    plot_data['Rank'],
    plot_data['Revenue']
)

ax1.set_xlabel('Customer Rank')
ax1.set_ylabel('Revenue')
ax1.set_title('Pareto Analysis - Top Customers')

# Cumulative percentage line
ax2 = ax1.twinx()

ax2.plot(
    plot_data['Rank'],
    plot_data['Cumulative_%'],
    marker='o'
)

ax2.axhline(
    y=80,
    linestyle='--'
)

ax2.set_ylabel('Cumulative Revenue %')

plt.xticks(plot_data['Rank'])

plt.tight_layout()
plt.savefig("Pareto_Chart.png", bbox_inches="tight")
plt.show()

In [ ]:
#ABC Classification
prod_rev = (
    df.groupby('Products')['Revenue']
      .sum()
      .sort_values(ascending=False)
      .reset_index()
)

prod_rev['Revenue_%'] = (
    prod_rev['Revenue']
    /
    prod_rev['Revenue'].sum()
) * 100

prod_rev['Cumulative_%'] = (
    prod_rev['Revenue_%']
    .cumsum()
)

prod_rev.head(10)

In [ ]:
#ABC Classification
def classify_abc(cum_pct):

    if cum_pct <= 70:
        return 'A'

    elif cum_pct <= 90:
        return 'B'

    else:
        return 'C'

prod_rev['ABC_Class'] = (
    prod_rev['Cumulative_%']
    .apply(classify_abc)
)

In [ ]:
#ABC Classification
abc_summary = prod_rev.groupby(
    'ABC_Class'
).agg(
    Products=('Products','count'),
    Total_Revenue=('Revenue','sum')
)

abc_summary

In [ ]:
#COHORT Analysis
df['Transaction_Month'] = (
    df['Date'].dt.to_period('M')
)

df['Cohort_Month'] = (
    df.groupby('Customer')['Date']
      .transform('min')
      .dt.to_period('M')
)

In [ ]:
#COHORT Analysis
def month_diff(row):

    return (
        (row['Transaction_Month'].year
         -
         row['Cohort_Month'].year) * 12
        +
        (row['Transaction_Month'].month
         -
         row['Cohort_Month'].month)
    )

df['Cohort_Index'] = df.apply(
    month_diff,
    axis=1
)

In [ ]:
#COHORT Analysis
cohort_data = (
    df.groupby(
        ['Cohort_Month',
         'Cohort_Index']
    )['Customer']
     .nunique()
     .reset_index()
)

cohort_counts = cohort_data.pivot(
    index='Cohort_Month',
    columns='Cohort_Index',
    values='Customer'
)

cohort_counts.head()

In [ ]:
#COHORT Analysis
cohort_size = cohort_counts.iloc[:,0]

retention_matrix = (
    cohort_counts.divide(
        cohort_size,
        axis=0
    ) * 100
)

retention_matrix.round(1)

In [ ]:
#COHORT Analysis
fig, ax = plt.subplots(
    figsize=(12,8)
)

data = retention_matrix.values

im = ax.imshow(
    data,
    cmap='YlGnBu',
    aspect='auto',
    vmin=0,
    vmax=100
)

plt.colorbar(im)
plt.savefig("Cohort_Heatmap.png", bbox_inches="tight")
plt.show()

In [ ]:
rfm.to_csv("RFM_Output.csv", index=False)

print("RFM saved")

In [ ]:
cust_rev.to_csv("Pareto_Output.csv", index=False)

print("Pareto saved")

In [ ]:
prod_rev.to_csv("ABC_Output.csv", index=False)

print("ABC saved")

In [ ]:
retention_matrix.to_csv("Cohort_Output.csv")

print("Cohort saved")

In [ ]:
import os

os.listdir()